# S6E7 — v7 : Target-Encoding + correction beta (recette souvik, GPU)

Différences vs v6 :

- **Features brutes uniquement** (NaN conservés) — le FE par stats de ligne n'apportait rien
- **Target-encoding des catégorielles** (fuit-proof : calculé in-fold pour l'OOF, plein-train pour le test), avec lissage vers le prior
- **Correction beta** `proba / prior**beta` calibrée sur l'OOF (balanced accuracy) — plus propre que les multiplicateurs libres
- XGBoost cuda 3 seeds × 5 folds (sur TE) + CatBoost GPU 3 seeds × 5 folds (cats natives), blend calibré OOF
- Sauvegarde `oof.npy` / `test_proba.npy` pour le méta-blend local avec C1

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
N_FOLDS = 5
SEEDS = [42, 2026, 777]
SMOOTH = 50.0  # lissage du target-encoding vers le prior

CANDIDATES = ["/kaggle/input/competitions/playground-series-s6e7",
              "/kaggle/input/playground-series-s6e7", "./data", "."]
DATA_DIR = next(p for p in CANDIDATES if os.path.exists(os.path.join(p, "train.csv")))
print("DATA_DIR:", DATA_DIR)

In [ ]:
TARGET = "health_condition"; ID_COL = "id"
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

cat_cols = [c for c in train.columns if train[c].dtype == "object" and c != TARGET]
num_cols = [c for c in train.columns if c not in [TARGET, ID_COL] + cat_cols]

target_le = LabelEncoder()
y = target_le.fit_transform(train[TARGET])
N_CLASSES = len(target_le.classes_)
prior = np.bincount(y) / len(y)
print("classes:", list(target_le.classes_), "| prior:", prior.round(4))

class_weight = len(y) / (N_CLASSES * np.bincount(y))
sample_weight = class_weight[y]

# cat -> codes entiers (pour CatBoost et le TE)
cat_codes = {}
for c in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[c], test[c]]).astype(str))
    cat_codes[c] = (le.transform(train[c].astype(str)), le.transform(test[c].astype(str)))

def te_fit_transform(codes_tr, y_tr, codes_va, n_classes, smooth=SMOOTH):
    """Target-encoding fuit-proof : stats sur y_tr, appliquées à tr(OOF exclu) et va."""
    K = codes_tr.max() + 1
    y1 = np.eye(n_classes)[y_tr]
    cnt = np.bincount(codes_tr, minlength=K)[:, None]
    sums = np.zeros((K, n_classes))
    for k in range(n_classes):
        sums[:, k] = np.bincount(codes_tr, weights=y1[:, k], minlength=K)
    pr = y1.mean(0)[None, :]
    te = (sums + smooth * pr) / (cnt + smooth)
    return te[codes_tr], te[codes_va]

X_num_tr = train[num_cols].to_numpy(dtype=float)
X_num_te = test[num_cols].to_numpy(dtype=float)
print("num:", X_num_tr.shape, "| cats:", len(cat_cols))

## CV — XGBoost sur TE + CatBoost natif, OOF partagé entre seeds

In [ ]:
oof_xgb = np.zeros((len(train), N_CLASSES))
oof_cat = np.zeros((len(train), N_CLASSES))
tst_xgb = np.zeros((len(test), N_CLASSES))
tst_cat = np.zeros((len(test), N_CLASSES))
n_runs = N_FOLDS * len(SEEDS)

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(skf.split(X_num_tr, y)):
        # --- TE fuit-proof pour ce fold ---
        te_tr_parts, te_va_parts, te_te_parts = [], [], []
        for c in cat_cols:
            codes_tr_full, codes_te = cat_codes[c]
            ctr, cva = codes_tr_full[tr], codes_tr_full[va]
            _, te_va = te_fit_transform(ctr, y[tr], cva, N_CLASSES)
            _, te_te = te_fit_transform(ctr, y[tr], codes_te, N_CLASSES)
            # pour le train on ne peut pas s'encoder soi-même -> TE laissé de côté, on utilise LOO approx :
            K = ctr.max() + 1
            y1 = np.eye(N_CLASSES)[y[tr]]
            cnt = np.bincount(ctr, minlength=K)[:, None]
            sums = np.stack([np.bincount(ctr, weights=y1[:, k], minlength=K) for k in range(N_CLASSES)], 1)
            pr = y1.mean(0)[None, :]
            # LOO : retirer sa propre contribution
            sums_loo = sums[ctr] - y1
            cnt_loo = cnt[ctr] - 1
            te_tr = (sums_loo + SMOOTH * pr) / (cnt_loo + SMOOTH)
            te_tr_parts.append(te_tr); te_va_parts.append(te_va); te_te_parts.append(te_te)
        Xtr = np.hstack([X_num_tr[tr]] + te_tr_parts)
        Xva = np.hstack([X_num_tr[va]] + te_va_parts)
        Xte = np.hstack([X_num_te] + te_te_parts)

        m = xgb.XGBClassifier(
            objective="multi:softprob", num_class=N_CLASSES,
            tree_method="hist", device="cuda",
            n_estimators=2000, learning_rate=0.05, max_depth=6, min_child_weight=1.0,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            early_stopping_rounds=200, random_state=seed, n_jobs=-1)
        m.fit(Xtr, y[tr], sample_weight=sample_weight[tr],
              eval_set=[(Xva, y[va])], verbose=False)
        oof_xgb[va] += m.predict_proba(Xva) / len(SEEDS)
        tst_xgb += m.predict_proba(Xte) / n_runs
        b = balanced_accuracy_score(y[va], m.predict_proba(Xva).argmax(1))
        print(f"[XGB s{seed}] fold {fold+1} | iter={m.best_iteration} | bal_acc={b:.5f}")
print("[XGB] OOF:", round(balanced_accuracy_score(y, oof_xgb.argmax(1)), 5))

In [ ]:
cat_idx = list(range(X_num_tr.shape[1], X_num_tr.shape[1] + len(cat_cols)))
Xc_tr = np.hstack([X_num_tr] + [cat_codes[c][0].reshape(-1, 1) for c in cat_cols])
Xc_te = np.hstack([X_num_te] + [cat_codes[c][1].reshape(-1, 1) for c in cat_cols])
feat_names = num_cols + cat_cols
df_tr = pd.DataFrame(Xc_tr, columns=feat_names)
df_te = pd.DataFrame(Xc_te, columns=feat_names)
for c in cat_cols:
    df_tr[c] = df_tr[c].astype(int)
    df_te[c] = df_te[c].astype(int)
cat_names = cat_cols

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(skf.split(df_tr, y)):
        m = CatBoostClassifier(
            loss_function="MultiClass", classes_count=N_CLASSES,
            iterations=5000, learning_rate=0.04, depth=8, l2_leaf_reg=3.0,
            class_weights=list(class_weight), cat_features=cat_names,
            task_type="GPU", devices="0", random_seed=seed, verbose=False,
            od_type="Iter", od_wait=250)
        m.fit(df_tr.iloc[tr], y[tr], eval_set=(df_tr.iloc[va], y[va]), use_best_model=True)
        oof_cat[va] += m.predict_proba(df_tr.iloc[va]) / len(SEEDS)
        tst_cat += m.predict_proba(df_te) / n_runs
        b = balanced_accuracy_score(y[va], m.predict_proba(df_tr.iloc[va]).argmax(1))
        print(f"[CAT s{seed}] fold {fold+1} | iter={m.best_iteration_} | bal_acc={b:.5f}")
print("[CAT] OOF:", round(balanced_accuracy_score(y, oof_cat.argmax(1)), 5))

## Blend + correction beta calibrés sur l'OOF

In [ ]:
def bacc(p):
    return balanced_accuracy_score(y, p.argmax(1))

best_w, best_s = 0.5, 0.0
for w in np.arange(0.0, 1.01, 0.05):
    s = bacc(w * oof_xgb + (1 - w) * oof_cat)
    if s > best_s:
        best_s, best_w = s, w
print(f"blend xgb={best_w:.2f} cat={1-best_w:.2f} | OOF = {best_s:.5f}")

oof = best_w * oof_xgb + (1 - best_w) * oof_cat
tst = best_w * tst_xgb + (1 - best_w) * tst_cat

BETAS = np.arange(0.0, 1.51, 0.05)
scores = {b: balanced_accuracy_score(y, (oof / prior ** b).argmax(1)) for b in BETAS}
best_beta = max(scores, key=scores.get)
print(f"beta optimal = {best_beta:.2f} | OOF {scores[0.0]:.5f} -> {scores[best_beta]:.5f}")

np.save("oof.npy", oof); np.save("test_proba.npy", tst)
with open("beta.txt", "w") as f:
    f.write(str(best_beta))

pred = target_le.inverse_transform((tst / prior ** best_beta).argmax(1))
submission = sample.copy()
submission[TARGET] = pred
submission.to_csv("submission.csv", index=False)
print(submission[TARGET].value_counts(normalize=True))
print("OK submission.csv")